# 02 · Core AI — precompute the features, generate answers live

**This is the heart of the workshop** and the one change from the current
prototype. Instead of calling `SUMMARIZE`/`COMPLETE` at query time (which forces
batching + truncation), we:

1. **enrich each row once** — sentiment + topic — as an incremental Dynamic Table, and
2. use **`AI_AGG` / `AI_SUMMARIZE_AGG`** for the live "why" summaries (they do the
   map-reduce for you, with no context-window limit).

You keep full flexibility: Cortex Analyst still writes varying SQL over the
enriched columns. You just stop paying to re-run the LLM on every question.

### Context

In [ ]:
SET sch = 'PLG_CORTEX_WORKSHOP.WS_' || REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9_]', '_');
USE WAREHOUSE PLG_WORKSHOP_WH;
USE SCHEMA IDENTIFIER($sch);

### FAST-PATH (only if you're behind)
Skips the build and creates `SURVEY_ENRICHED` from the answer key so you can keep
going. **Skip this cell if you're doing the hands-on build below.**
```sql
EXECUTE IMMEDIATE FROM @PLG_CORTEX_WORKSHOP.PUBLIC.WORKSHOP_STAGE/scaffold.sql;
```
_(If your facilitator hasn't staged scaffold.sql, just copy sections 2.2 and 2.4
from `reference/scaffold.sql` into a cell and run them.)_

### 1. See the token cost BEFORE reduction

In [ ]:
SELECT
  COUNT(*)                                                          AS n_rows,
  SUM(SNOWFLAKE.CORTEX.COUNT_TOKENS('llama3.1-8b', clarity_comment)) AS total_tokens
FROM SURVEY_RESPONSES;

### 2. Drop the junk with `AI_FILTER` (in plain language)
This replaces the hand-written "remove nee / nvt / emoji-only" rules with one
natural-language condition. We filter over `SURVEY_BASE` so `game_round_performance`
stays on each row.

In [ ]:
CREATE OR REPLACE VIEW SURVEY_CLEAN AS
SELECT *
FROM SURVEY_BASE
WHERE clarity_comment IS NOT NULL
  AND AI_FILTER(PROMPT('Is this a substantive comment about an email, not an empty or throwaway answer: {0}', clarity_comment));

### 3. Token cost AFTER reduction — compare to step 1

In [ ]:
SELECT
  COUNT(*)                                                          AS n_rows,
  SUM(SNOWFLAKE.CORTEX.COUNT_TOKENS('llama3.1-8b', clarity_comment)) AS total_tokens
FROM SURVEY_CLEAN;

### 4. THE PIVOT — enrich once, incrementally

**Fill the blank:** add a `clarity_topic` column using `AI_CLASSIFY` against the
fixed taxonomy. Because the AI functions live in the SELECT of a **Dynamic Table**,
an incremental refresh only reruns them on NEW rows — not the whole history.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE SURVEY_ENRICHED
  TARGET_LAG = '1 hour'
  WAREHOUSE  = PLG_WORKSHOP_WH
AS
SELECT
  s.response_id, s.player_id, s.brand, s.email_type, s.survey_date,
  s.game_round_performance, s.clarity_rating, s.tone_rating,
  s.clarity_comment, s.tone_comment,
  AI_SENTIMENT(s.clarity_comment):categories[0]:sentiment::string AS clarity_sentiment,
  AI_SENTIMENT(s.tone_comment):categories[0]:sentiment::string    AS tone_sentiment
  -- >>> YOUR PART <<< : add clarity_topic using
  --   AI_CLASSIFY(s.clarity_comment,
  --     ['content_clarity','layout','too_long','tone','technical','pricing','other']
  --   ):labels[0]::string AS clarity_topic
FROM SURVEY_CLEAN s;

_Answer key: `reference/scaffold.sql` section 2.4._

### 5. Verify the enrichment on a sample

In [ ]:
SELECT clarity_comment, clarity_sentiment, clarity_topic
FROM SURVEY_ENRICHED
LIMIT 10;

### 6. Live synthesis done right — `AI_AGG`
No batches of 50, no `LEFT()` truncation. `AI_AGG` maps over every row and
reduces for you. `game_round_performance` is now a plain column.

In [ ]:
SELECT
  game_round_performance,
  AI_AGG(clarity_comment,
    'Summarise in English the top 3 recurring complaints about email clarity in these Dutch comments. Return a short bullet list.') AS top_clarity_issues
FROM SURVEY_ENRICHED
WHERE game_round_performance = 'poor'
GROUP BY game_round_performance;

### Checkpoint ✅ — say it in your own words
Be able to answer: **why do we precompute `clarity_sentiment` and `clarity_topic`,
but NOT precompute the `AI_AGG` summary?**

> Because sentiment/topic are stable per row (compute once, reuse in every query),
> while the summary depends on the question being asked (which rows, what to
> highlight) so it belongs at query time. That split is the whole cost story.

Next: `03_advanced_ai_search.ipynb` (stretch) or jump to `04_consumption_eval.ipynb`.